## Adaptive pretraining

### Colab Setup

In [ ]:
import os
import subprocess
import sys

# local runs: the repo root is one level up. Colab chdirs there below.
sys.path.insert(0, "..")

# On Colab: clone the repo, install deps, mount Drive for results.csv. The repo is
# public, so no token. Python caches imports -- restart the runtime after any code
# change, or the clone refreshes and the old module stays loaded.
REPO = "https://github.com/IronQuant/mlds_codebase.git"
ROOT = "/content/mlds_codebase"

if "google.colab" in sys.modules:
    if os.path.isdir(ROOT):
        subprocess.run(["git", "-C", ROOT, "fetch", "-q", "origin"], check=True)
        subprocess.run(
            ["git", "-C", ROOT, "reset", "--hard", "-q", "origin/main"], check=True
        )
    else:
        subprocess.run(["git", "clone", "-q", REPO, ROOT], check=True)

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "transformers>=4.48",
            "ftfy",
            "nltk",
            "polars",
            "fastexcel",
            "sentencepiece",
            "protobuf",
        ],
        check=True,
    )
    os.chdir(ROOT)
    sys.path.insert(0, ROOT)

    from google.colab import drive

    drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Key Imports

In [ ]:
import pandas as pd
import torch

from config import APT, APT_EPOCHS, IDIOMS, RESULTS_DIR, SHAH_PLM, SHAH_SEEDS
from data.apt_pools import build_eval_set, build_pools, build_tapt_pool
from data.loader_twd_labelled import load_splits
from models.apt import adapt
from models.frozen_probe import probe
from models.plm_finetune import finetune
from sklearn.metrics import f1_score

from utils.results import already_done, save_result

OUT = RESULTS_DIR / "results.csv"
ENC = "roberta-large"
SEEDS = SHAH_SEEDS
FORCE = False
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("results ->", OUT, "| device:", DEVICE)
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0))

results -> /content/drive/MyDrive/thesis/results.csv | device: cuda
NVIDIA A100-SXM4-40GB


### Arms

In [ ]:
VANILLA = SHAH_PLM[ENC]["model_name"]
_pools = build_pools(verbose=False)
FOMC_POOL = _pools[0]["sentence"].to_list()
GLOBAL_POOL = _pools[1]["sentence"].to_list()
print(f"fomc pool: {len(FOMC_POOL):,} | global pool: {len(GLOBAL_POOL):,}")

# arm -> (sentences, epochs key, starting checkpoint, seed-dependent)
ARMS = {
    "dapt-fomc": (lambda seed: FOMC_POOL, "dapt", VANILLA, False),
    "dapt-global": (lambda seed: GLOBAL_POOL, "dapt", VANILLA, False),
    "tapt": (
        lambda seed: load_splits("benchmark", seed=seed)[0]["sentence"].to_list(),
        "tapt",
        VANILLA,
        True,
    ),
    # Gururangan's headline combined setting: DAPT then TAPT on the task's own
    # training text, starting from the already-adapted global checkpoint
    "dapt-global+tapt": (
        lambda seed: load_splits("benchmark", seed=seed)[0]["sentence"].to_list(),
        "tapt",
        str(RESULTS_DIR / "models" / "dapt-global"),
        True,
    ),
}


downloading TDW repo tarball (~61MB)...
extracted -> /tmp/tmphkvsfyfk
  meeting_minutes: 230 docs -> 47,340 sentences
  speech: 1026 docs -> 107,548 sentences
  press_conference: 63 docs -> 24,750 sentences


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Seed 5768 | Train: 1984 rows  |  Test: 496 rows  |  Total: 2480
downloading TDW repo tarball (~61MB)...
extracted -> /tmp/tmpsnz8xe5t
  meeting_minutes: 214 docs -> 20,618 sentences
  press_conference: 63 docs -> 5,086 sentences
  speech: 201 docs -> 12,465 sentences
Seed 5768 | Train: 1984 rows  |  Test: 496 rows  |  Total: 2480
fomc pool: 163,052 sentences


### Continued pretraining

In [ ]:
for arm, (pool_fn, epochs_key, start, per_seed) in ARMS.items():
    for seed in SEEDS if per_seed else [None]:
        name = f"{arm}-s{seed}" if per_seed else arm
        save_dir = str(RESULTS_DIR / "models" / name)
        if os.path.isdir(save_dir):
            print(f"{name}: already adapted, skipping")
            continue
        sentences = pool_fn(seed)
        print(f"{name}: {len(sentences):,} sentences", flush=True)
        adapt(
            sentences,
            model_name=start,
            epochs=APT_EPOCHS[epochs_key],
            save_dir=save_dir,
            device=DEVICE,
            verbose=True,
            **APT,
        )

dapt-fomc: already adapted, skipping
tapt-s5768: already adapted, skipping
tapt-s78516: already adapted, skipping
tapt-s944601: already adapted, skipping
curated-tapt-s5768: already adapted, skipping
curated-tapt-s78516: already adapted, skipping
downloading TDW repo tarball (~61MB)...
extracted -> /tmp/tmp5klh9mgm
  meeting_minutes: 214 docs -> 20,618 sentences
  press_conference: 63 docs -> 5,086 sentences
  speech: 201 docs -> 12,465 sentences
Seed 944601 | Train: 1984 rows  |  Test: 496 rows  |  Total: 2480
downloading TDW repo tarball (~61MB)...
extracted -> /tmp/tmp57rwg1hx
  meeting_minutes: 214 docs -> 20,618 sentences
  press_conference: 63 docs -> 5,086 sentences
  speech: 201 docs -> 12,465 sentences
Seed 5768 | Train: 1984 rows  |  Test: 496 rows  |  Total: 2480
tapt pool seed 944601: 35,257 -> 33,146
curated-tapt-s944601: 33,146 sentences


Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

    tokenizing 33,146 sentences...
    training: 1,036 batches/epoch x 12 epoch(s), 12,432 updates at effective batch 32
    100/1,036: mlm loss 1.3425 | 2.5 it/s, eta 81 min
    200/1,036: mlm loss 1.3234 | 2.6 it/s, eta 80 min
    300/1,036: mlm loss 1.2989 | 2.6 it/s, eta 79 min
    400/1,036: mlm loss 1.2709 | 2.5 it/s, eta 79 min
    500/1,036: mlm loss 1.2583 | 2.6 it/s, eta 78 min
    600/1,036: mlm loss 1.2430 | 2.6 it/s, eta 77 min
    700/1,036: mlm loss 1.2317 | 2.6 it/s, eta 76 min
    800/1,036: mlm loss 1.2223 | 2.6 it/s, eta 75 min
    900/1,036: mlm loss 1.2170 | 2.6 it/s, eta 75 min
    1,000/1,036: mlm loss 1.2136 | 2.5 it/s, eta 75 min
    epoch 0: mlm loss 1.2095
    100/1,036: mlm loss 1.1194 | 2.5 it/s, eta 76 min
    200/1,036: mlm loss 1.1067 | 2.5 it/s, eta 76 min
    300/1,036: mlm loss 1.1070 | 2.5 it/s, eta 74 min
    400/1,036: mlm loss 1.0917 | 2.5 it/s, eta 73 min
    500/1,036: mlm loss 1.0864 | 2.5 it/s, eta 72 min
    600/1,036: mlm loss 1.0864 | 2.5 i

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

    saved -> /content/drive/MyDrive/thesis/models/curated-tapt-s944601
downloading TDW repo tarball (~61MB)...
extracted -> /tmp/tmp2wpjkeb1
  meeting_minutes: 214 docs -> 20,618 sentences
  press_conference: 63 docs -> 5,086 sentences
  speech: 201 docs -> 12,465 sentences
Seed 5768 | Train: 1984 rows  |  Test: 496 rows  |  Total: 2480
downloading TDW repo tarball (~61MB)...
extracted -> /tmp/tmp176yu0ox
  meeting_minutes: 214 docs -> 20,618 sentences
  press_conference: 63 docs -> 5,086 sentences
  speech: 201 docs -> 12,465 sentences
Seed 5768 | Train: 1984 rows  |  Test: 496 rows  |  Total: 2480
tapt pool seed 5768: 35,257 -> 33,148
dapt-fomc+curated-tapt-s5768: 33,148 sentences


...

    saved -> /content/drive/MyDrive/thesis/models/dapt-fomc+curated-tapt-s78516
downloading TDW repo tarball (~61MB)...
extracted -> /tmp/tmpjr4j4kkp
  meeting_minutes: 214 docs -> 20,618 sentences
  press_conference: 63 docs -> 5,086 sentences
  speech: 201 docs -> 12,465 sentences
Seed 944601 | Train: 1984 rows  |  Test: 496 rows  |  Total: 2480
downloading TDW repo tarball (~61MB)...
extracted -> /tmp/tmpe_7v_ruq
  meeting_minutes: 214 docs -> 20,618 sentences
  press_conference: 63 docs -> 5,086 sentences
  speech: 201 docs -> 12,465 sentences
Seed 5768 | Train: 1984 rows  |  Test: 496 rows  |  Total: 2480
tapt pool seed 944601: 35,257 -> 33,146
dapt-fomc+curated-tapt-s944601: 33,146 sentences


Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

    tokenizing 33,146 sentences...
    training: 1,036 batches/epoch x 12 epoch(s), 12,432 updates at effective batch 32
    100/1,036: mlm loss 1.0215 | 2.6 it/s, eta 80 min
    200/1,036: mlm loss 1.0341 | 2.6 it/s, eta 79 min
    300/1,036: mlm loss 1.0318 | 2.6 it/s, eta 79 min
    400/1,036: mlm loss 1.0204 | 2.6 it/s, eta 79 min
    500/1,036: mlm loss 1.0232 | 2.6 it/s, eta 77 min
    600/1,036: mlm loss 1.0220 | 2.6 it/s, eta 77 min
    700/1,036: mlm loss 1.0222 | 2.6 it/s, eta 76 min
    800/1,036: mlm loss 1.0223 | 2.6 it/s, eta 75 min
    900/1,036: mlm loss 1.0229 | 2.6 it/s, eta 75 min
    1,000/1,036: mlm loss 1.0250 | 2.5 it/s, eta 75 min
    epoch 0: mlm loss 1.0243
    100/1,036: mlm loss 1.0184 | 2.5 it/s, eta 77 min
    200/1,036: mlm loss 1.0032 | 2.5 it/s, eta 76 min
    300/1,036: mlm loss 1.0067 | 2.5 it/s, eta 74 min
    400/1,036: mlm loss 1.0020 | 2.5 it/s, eta 73 min
    500/1,036: mlm loss 1.0011 | 2.5 it/s, eta 72 min
    600/1,036: mlm loss 1.0034 | 2.5 i

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

    saved -> /content/drive/MyDrive/thesis/models/dapt-fomc+curated-tapt-s944601


### Held-out MLM loss

In [ ]:
# a fixed slice of the filtered corpus, subtracted from every pool and holding no
# labelled sentence, so no arm has trained on it and it needs no seed.
eval_on = build_eval_set()["sentence"].to_list()

from torch.utils.data import DataLoader
from transformers import (
    AutoModelForMaskedLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
)



def mlm_loss(model_path, seed=0):
    torch.manual_seed(seed)
    tok = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForMaskedLM.from_pretrained(model_path).to(DEVICE).eval()
    enc = tok(eval_on, truncation=True, max_length=APT["max_len"])
    rows = [
        {"input_ids": i, "attention_mask": m}
        for i, m in zip(enc["input_ids"], enc["attention_mask"])
    ]
    dl = DataLoader(
        rows,
        batch_size=APT["batch_size"],
        collate_fn=DataCollatorForLanguageModeling(
            tok, mlm_probability=APT["mlm_probability"]
        ),
    )
    total, n = 0.0, 0
    with torch.no_grad():
        for batch in dl:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            total += model(**batch).loss.item()
            n += 1
    del model
    torch.cuda.empty_cache()
    return total / n


paths = {ENC: VANILLA}
for arm, (_, _, _, per_seed) in ARMS.items():
    for seed in SEEDS if per_seed else [None]:
        name = f"{arm}-s{seed}" if per_seed else arm
        path = str(RESULTS_DIR / "models" / name)
        if os.path.isdir(path):
            paths[name] = path

rows = [dict(model=k, mlm_loss=round(mlm_loss(v), 4)) for k, v in paths.items()]
for r in rows:
    print(f"{r['model']}: {r['mlm_loss']}")

pd.DataFrame(rows).to_csv(RESULTS_DIR / "mlm_loss.csv", index=False)
print("saved ->", RESULTS_DIR / "mlm_loss.csv")


downloading TDW repo tarball (~61MB)...
extracted -> /tmp/tmpx4hyl37r
  meeting_minutes: 214 docs -> 20,618 sentences
  press_conference: 63 docs -> 5,086 sentences
  speech: 201 docs -> 12,465 sentences
Seed 5768 | Train: 1984 rows  |  Test: 496 rows  |  Total: 2480
eval set: 1,652 of 33,031 unlabelled filtered


Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

roberta-large: 1.2529
dapt-fomc: 0.9524
tapt-s5768: 1.0716
tapt-s78516: 1.0668
tapt-s944601: 1.0652
curated-tapt-s5768: 0.8173
curated-tapt-s78516: 0.8159
curated-tapt-s944601: 0.8238
dapt-fomc+curated-tapt-s5768: 0.8122
dapt-fomc+curated-tapt-s78516: 0.8027
dapt-fomc+curated-tapt-s944601: 0.8092
saved -> /content/drive/MyDrive/thesis/mlm_loss.csv


### Masked-idiom probe

In [ ]:
from transformers import pipeline


def probe_idioms(model_path):
    mlm = pipeline("fill-mask", model=model_path, device=0 if DEVICE == "cuda" else -1)
    mask = mlm.tokenizer.mask_token
    rows = []
    for phrase, gold in IDIOMS:
        top5 = [
            r["token_str"].strip().lower()
            for r in mlm(phrase.replace("[MASK]", mask), top_k=5)
        ]
        rows.append(
            dict(phrase=phrase, gold=gold, hit=gold.lower() in top5, top5="|".join(top5))
        )
    del mlm
    torch.cuda.empty_cache()
    return rows


models = {ENC: VANILLA}
for arm, (_, _, _, per_seed) in ARMS.items():
    name = f"{arm}-s{SEEDS[0]}" if per_seed else arm
    models[arm] = str(RESULTS_DIR / "models" / name)

records = []
for label, path in models.items():
    rows = probe_idioms(path)
    records += [dict(model=label, **r) for r in rows]
    print(f"{label}: {sum(r['hit'] for r in rows)}/{len(rows)}", flush=True)

idf = pd.DataFrame(records)
idf.to_csv(RESULTS_DIR / "idioms.csv", index=False)
print("saved ->", RESULTS_DIR / "idioms.csv")

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


roberta-large: 69/100


Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

dapt-fomc: 83/100


...

tapt: 78/100


Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

curated-tapt: 81/100


Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

dapt-fomc+curated-tapt: 83/100
saved -> /content/drive/MyDrive/thesis/idioms.csv


### Fine-tune

In [ ]:
cfg = SHAH_PLM[ENC]

for arm, (_, _, _, per_seed) in ARMS.items():
    for seed in SEEDS:
        model_key = f"{arm}:{ENC}"
        if already_done(OUT, force=FORCE, model=model_key, corpus="twd", seed=seed):
            print(f"{model_key} seed {seed}: already done, skipping")
            continue
        name = f"{arm}-s{seed}" if per_seed else arm
        train, test = load_splits("benchmark", seed=seed)
        model, tok_, metrics = finetune(
            train,
            model_name=str(RESULTS_DIR / "models" / name),
            lr=cfg["lr"],
            batch_size=cfg["batch_size"],
            seed=seed,
            test_df=test,
            device=DEVICE,
            verbose=True,
        )
        save_result(
            OUT,
            model=model_key,
            corpus="twd",
            seed=seed,
            epochs=metrics["epochs"],
            weighted_f1=round(metrics["test_f1"], 4),
            macro_f1=round(metrics["test_macro_f1"], 4),
        )
        print(f"{model_key} seed {seed}: macro={metrics['test_macro_f1']:.4f}")
        del model, tok_
        torch.cuda.empty_cache()

Seed 5768 | Train: 1984 rows  |  Test: 496 rows  |  Total: 2480


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: /content/drive/MyDrive/thesis/models/dapt-fomc
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


    epoch  0: val CE=1.0944  acc=0.4949  wF1=0.3423  mF1=0.2518  es=0  20.7s
    epoch  1: val CE=0.8370  acc=0.6389  wF1=0.6426  mF1=0.6236  es=0  20.4s
    epoch  2: val CE=0.9244  acc=0.5833  wF1=0.5852  mF1=0.5859  es=1  19.6s
    epoch  3: val CE=0.7931  acc=0.6869  wF1=0.6904  mF1=0.6872  es=0  19.7s
    epoch  4: val CE=0.7805  acc=0.7273  wF1=0.7267  mF1=0.7103  es=0  20.2s
    epoch  5: val CE=0.8879  acc=0.6869  wF1=0.6898  mF1=0.6824  es=1  19.9s
    epoch  6: val CE=0.9038  acc=0.6818  wF1=0.6855  mF1=0.6770  es=2  19.6s
    epoch  7: val CE=1.1532  acc=0.6995  wF1=0.6987  mF1=0.6846  es=3  19.7s
    epoch  8: val CE=1.1804  acc=0.7273  wF1=0.7280  mF1=0.7128  es=0  20.2s
    epoch  9: val CE=1.3763  acc=0.6995  wF1=0.7029  mF1=0.6942  es=1  19.6s
    epoch 10: val CE=1.2823  acc=0.6894  wF1=0.6925  mF1=0.6850  es=2  19.8s
    epoch 11: val CE=1.3834  acc=0.7273  wF1=0.7274  mF1=0.7130  es=0  20.1s
    epoch 12: val CE=1.3107  acc=0.7172  wF1=0.7203  mF1=0.7110  es=1  19.8s

...

[transformers] RobertaForSequenceClassification LOAD REPORT from: /content/drive/MyDrive/thesis/models/dapt-fomc+curated-tapt-s78516
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


    epoch  0: val CE=1.0454  acc=0.4823  wF1=0.4679  mF1=0.3985  es=0  19.3s
    epoch  1: val CE=0.8250  acc=0.6364  wF1=0.6454  mF1=0.6198  es=0  19.6s
    epoch  2: val CE=0.7399  acc=0.6894  wF1=0.6956  mF1=0.6758  es=0  19.5s
    epoch  3: val CE=0.7285  acc=0.6818  wF1=0.6880  mF1=0.6665  es=1  19.1s
    epoch  4: val CE=0.8278  acc=0.7323  wF1=0.7368  mF1=0.7086  es=0  19.4s
    epoch  5: val CE=1.0032  acc=0.7197  wF1=0.7232  mF1=0.6957  es=1  19.1s
    epoch  6: val CE=1.2159  acc=0.7348  wF1=0.7318  mF1=0.7036  es=2  19.1s
    epoch  7: val CE=1.2934  acc=0.7121  wF1=0.7179  mF1=0.6878  es=3  18.9s
    epoch  8: val CE=1.2609  acc=0.7323  wF1=0.7365  mF1=0.7086  es=0  19.4s
    epoch  9: val CE=1.3785  acc=0.6970  wF1=0.7033  mF1=0.6864  es=1  18.8s
    epoch 10: val CE=1.3763  acc=0.7121  wF1=0.7177  mF1=0.6897  es=2  19.1s
    epoch 11: val CE=1.6901  acc=0.6970  wF1=0.7050  mF1=0.6728  es=3  19.0s
    epoch 12: val CE=1.6483  acc=0.7273  wF1=0.7223  mF1=0.6917  es=4  19.1s

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: /content/drive/MyDrive/thesis/models/dapt-fomc+curated-tapt-s944601
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


    epoch  0: val CE=0.9307  acc=0.5985  wF1=0.6031  mF1=0.5864  es=0  20.0s
    epoch  1: val CE=0.7245  acc=0.6793  wF1=0.6838  mF1=0.6722  es=0  20.0s
    epoch  2: val CE=0.7524  acc=0.7071  wF1=0.7121  mF1=0.6991  es=0  19.9s
    epoch  3: val CE=0.8798  acc=0.7374  wF1=0.7383  mF1=0.7292  es=0  19.6s
    epoch  4: val CE=1.0442  acc=0.7071  wF1=0.7103  mF1=0.7017  es=1  19.4s
    epoch  5: val CE=1.1393  acc=0.7045  wF1=0.7081  mF1=0.6932  es=2  19.5s
    epoch  6: val CE=1.2154  acc=0.7222  wF1=0.7241  mF1=0.7091  es=3  19.3s
    epoch  7: val CE=1.3451  acc=0.7020  wF1=0.7052  mF1=0.6936  es=4  19.1s
    epoch  8: val CE=1.4476  acc=0.7323  wF1=0.7332  mF1=0.7180  es=5  19.3s
    epoch  9: val CE=1.3849  acc=0.7146  wF1=0.7183  mF1=0.7049  es=6  19.3s
    epoch 10: val CE=1.4636  acc=0.7071  wF1=0.7073  mF1=0.6921  es=7  19.3s
dapt-fomc+curated-tapt:roberta-large seed 944601: macro=0.7177


### Frozen probe

In [ ]:
for arm, (_, _, _, per_seed) in ARMS.items():
    for seed in SEEDS:
        model_key = f"frozen-{arm}:{ENC}"
        if already_done(OUT, force=FORCE, model=model_key, corpus="twd", seed=seed):
            print(f"{model_key} seed {seed}: already done, skipping")
            continue
        name = f"{arm}-s{seed}" if per_seed else arm
        train, test = load_splits("benchmark", seed=seed)
        pred = probe(
            train,
            test,
            model_name=str(RESULTS_DIR / "models" / name),
            device=DEVICE,
            seed=seed,
        )
        true = test["label"].to_list()
        save_result(
            OUT,
            model=model_key,
            corpus="twd",
            seed=seed,
            epochs="",
            weighted_f1=round(f1_score(true, pred, average="weighted"), 4),
            macro_f1=round(f1_score(true, pred, average="macro"), 4),
        )
        print(f"{model_key} seed {seed}: macro={f1_score(true, pred, average='macro'):.4f}")


Seed 5768 | Train: 1984 rows  |  Test: 496 rows  |  Total: 2480


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: /content/drive/MyDrive/thesis/models/dapt-fomc
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


frozen-dapt-fomc:roberta-large seed 5768: macro=0.6169
Seed 78516 | Train: 1984 rows  |  Test: 496 rows  |  Total: 2480


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

...

[transformers] RobertaModel LOAD REPORT from: /content/drive/MyDrive/thesis/models/dapt-fomc+curated-tapt-s78516
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


frozen-dapt-fomc+curated-tapt:roberta-large seed 78516: macro=0.6280
Seed 944601 | Train: 1984 rows  |  Test: 496 rows  |  Total: 2480


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: /content/drive/MyDrive/thesis/models/dapt-fomc+curated-tapt-s944601
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


frozen-dapt-fomc+curated-tapt:roberta-large seed 944601: macro=0.6199
